# Cross Disease Pancreas

Normal mouse pancreas reference to combined disease-state query. Source: `MPNN_SelfDistill/benchmark/cross_disease_states/exp/train_mouse_pancreas.ipynb`. Only the original training and metric steps are retained; figure generation is omitted. Package imports, extracted-data paths, run output paths, and the CUDA device are the only training-cell adaptations.

Original training configuration: GNN=sage, batch_size=1054, max_steps=1000, high=True, rate=0.3.

Extract the matching restricted Zenodo archive into `data/experiments/` before running. Use `SCPATH_STUDY_DATA` to point to a different extraction directory and `SCPATH_DEVICE` to select a GPU. Set `PYTHONHASHSEED=42` before starting Jupyter, because the source preprocessing uses set-based gene intersections. Run cells in order with the `scPath` kernel. Historical result files in the archive are for comparison only; this notebook writes new outputs under `output/experiments/`.


In [ ]:
import os
import sys
from pathlib import Path
repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scpath/models.py").is_file())
sys.path.insert(0, str(repo_root))
data_root = Path(os.environ.get("SCPATH_STUDY_DATA", str(repo_root / "data/experiments")))
study_data = data_root / "cross_disease/original_paths/data"
run_dir = repo_root / "output/experiments/cross_disease_pancreas"
if not study_data.is_dir():
    raise FileNotFoundError(f"Extract the cross_disease_pancreas Zenodo archive under {data_root}")
run_dir.mkdir(parents=True, exist_ok=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required for this historical training workflow")
print("GPU:", torch.cuda.get_device_name(torch.device(os.environ.get("SCPATH_DEVICE", "cuda:0"))))


In [ ]:


import sys
sys.path.insert(0, str(repo_root))
from scpath.utils.data_preprocess import data_processing
from scpath.utils.pathway import built_pathway_graphs
from scpath.utils.data_preprocess import preprocess
from scpath.train import train
import torch
import os
from scpath.config import Config
import pandas as pd
import scipy.sparse as sparse
from scipy.io import mmread, mmwrite

train_path = str(study_data / "normal") + "/"  
test_path = str(study_data / "test_combined") + "/"  
path_output = str(run_dir / "output") + "/"
species_train='mouse'
species_test='mouse'
plot=False
save_data=False
os.makedirs(path_output, exist_ok=True)

adata_train_rna,adata_test_rna=data_processing(train_path,test_path,path_output,species_train,species_test,plot,save_data)


pathway_names=['Reactome','KEGG']  
file_path = str(repo_root / "data/pathway/mouse") + "/"  
built_pathway_graphs(file_path,pathway_names,adata_train_rna,adata_test_rna,path_output,n_neighbors=5)

In [ ]:
adata_train=adata_train_rna.copy()
adata_test=adata_test_rna.copy()

In [ ]:

n_train = adata_train.n_obs
n_test = adata_test.n_obs
total_samples = n_train + n_test
    

gnn_type='sage'
batch_size=1054
max_epochs=1000
print(f"Training strategy: GNN={gnn_type} | batch={batch_size} | epochs={max_epochs}")
    

effective_steps = max_epochs
adata_train, hvg_genes = preprocess(adata_train)
adata_test = preprocess(adata_test, hvg_genes=hvg_genes)

device = torch.device(os.environ.get("SCPATH_DEVICE", "cuda:0"))
config = Config(str(run_dir) + "/",adata_train.X.shape[1],device,True,batch_size=int(batch_size),gnn_type=gnn_type,max_steps=effective_steps,K=len(adata_test.obs['cell_type'].unique()),rate=0.3)
print(f"batch_size type: {type(config.batch_size)}")
train(config,adata_train,adata_test,pathway_names)

In [ ]:
import scanpy as sc
import pandas as pd
adata_train_result = sc.read_h5ad(run_dir / "logs/train_results.h5ad")  
adata_test_result = sc.read_h5ad(run_dir / "logs/test_results_distilled.h5ad")  

In [ ]:
import scanpy as sc
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, recall_score, classification_report

true_labels = adata_test_result.obs['labels']
pred_labels = adata_test_result.obs['final_distill_preds']


accuracy = accuracy_score(true_labels, pred_labels)
f1 = f1_score(true_labels, pred_labels, average='macro')

print(f"Overall accuracy: {accuracy:.4f}")
print(f"Macro F1: {f1:.4f}")

print(pd.DataFrame(classification_report(true_labels, pred_labels,output_dict=True)))


metrics_df = pd.DataFrame({
    'Accuracy': [accuracy],
    'F1_Score': [f1]
})
print(metrics_df)